# IS630: Green Light for an Indie Studio
## Notebook 07: Final Test Scoring
**Group**: [Insert names]

> **AI Coding Assistant Declaration**:
> An AI coding assistant drafted this specification at the group's direction. The group must review and accept it before code is written. All interpretations, evaluations of model performance and conclusions will be written by group members in the designated cells.

## Specification
*Written by AI Assistant*

Status: spec only, no code yet. This is the only notebook allowed to load `data/interim/00_test_holdout.parquet` (1,320 games, never seen by any earlier stage). It scores the models chosen in notebooks 03 to 06 once. Nothing is refitted, reselected or tuned after the holdout is loaded, and a disappointing number is reported as it is.

**Reads:** `data/interim/00_train.parquet`, `data/interim/00_test_holdout.parquet`, `data/processed/02_feature_spec.json`, `data/processed/02_train.parquet`, `data/processed/02_validation.parquet`, and the validation tables from 03, 04 and 06 for comparison
**Writes:** `data/interim/07_test_predictions.parquet`, tables `outputs/tables/07_test_*.csv`, figures `outputs/figures/07_test_*.png`

### 0. Prerequisite: move notebook 02's feature code into `src/is630`
- Notebook 02 builds every feature inline (cells 15 to 37), so the holdout cannot go through the same steps without copying them, which AGENTS.md forbids.
- Move them into `src/is630/features.py` as two functions. `build_features(df)` does everything that looks at one row only: price transforms, flags, genre and category encoding, language parsing. `apply_training_stats(df, stats)` does the two steps that use training data: the median language count used for imputation, and each developer's game count. `stats` comes from the training data and is saved in `02_feature_spec.json`.
- Notebook 02 then calls these functions. Check: 02's saved parquet files are identical before and after the move, as the 03 refactor was.

### 1. Freeze the choices before loading the holdout
- The first cell, run before the holdout is read, writes `outputs/tables/07_test_frozen_choices.csv`: the models to score, their predictors, priors, thresholds and seed, and the validation results they are being compared against. Its commit hash is recorded, so anyone can check the choices were not changed after seeing test results.
- Models scored (decision 1 confirms the list):
  - AQ2: baseline, OLS, and the Bayesian hierarchical model (notebook 03).
  - AQ3: the controllable model (notebook 04's reduced model).
  - AQ5: the Beta-Binomial reception model (notebook 06).

### 2. Refit on all training data
- The models in 03 to 06 were fitted on the development split (80% of training) and checked on validation (20%). Before scoring, each is refitted on development and validation together, the full `00_train`, with the specification unchanged. More data, same model.
- Standardisation, imputation and developer game counts are recomputed on the full training data, then applied to the holdout.
- Bayesian refits report the same convergence checks as before. A refit that does not converge is reported and not scored.
- Decision 2: refit on all training data (recommended), or score the development-split fits as they are.

### 3. Build the holdout
- Apply `build_features` and `apply_training_stats` to the holdout, then the same eligibility rules from notebook 02: `flag_completion_model` for AQ2 and AQ3, `flag_reception_model` for AQ5. Print rows in and out.
- Holdout developers seen in training get their developer effect. Unseen developers get zero, as in validation.

### 4. Score once
- AQ2 and AQ3: RMSE, MAE, R2 against the baseline, typical factor error, and 90% interval coverage and width, the same metrics as notebook 03. A developer-level bootstrap interval on the gain over the baseline, as in notebook 03.
- AQ5: calibration by predicted decile and 90% interval coverage by review-count band, the same as notebook 06.
- Output: `07_test_completion_scores.csv`, `07_test_reception_scores.csv`.

### 5. Test against validation
- One table per AQ with the validation number and the test number side by side for every metric.
- The comparison is reported, never acted on. If test is worse than validation, that is the finding.
- Output: `07_test_vs_validation.csv`, `07_test_vs_validation.png`.

Every result gets a blank interpretation cell.

### Decisions for the group before code is written
1. Which models are final, so they can be frozen in section 1. Proposed: all of those listed, since none was chosen by comparing against each other on validation in a way that would bias the test.
2. Refit on all training data before scoring, or score the development-split fits.
3. When to run it. It should run once, after notebooks 00 to 06 are final and the group has written the interpretations. Re-running after any upstream change means the test is no longer untouched, which has to be disclosed.